# Volatility targeting 2.0

Study volatility persistence, causal risk sizing and multi-asset allocation on real ETF data. This notebook exposes the estimator and control implementations, stages the experiment, and saves every trial. The acquisition, forecasts, allocation, execution and analysis code are contained in this notebook.

## 0. Competition rules and research contract
The shorter of the latest 20% of sessions and two years is locked. Research cannot acquire OOS. All outputs are net of costs; close-derived signals fill next close. Failed variants and reused validation remain disclosed.

## 1. Hypothesis and pre-registration
The committed specification takes precedence over ambiguous prose in the supplied guide. Twelve primary variants; diagnostic reruns are additionally disclosed.

In [ ]:
import csv
import hashlib
import importlib.metadata
import json
import os
import platform
import subprocess
import sys
import time
from dataclasses import asdict, dataclass, replace
from datetime import UTC, datetime
from pathlib import Path
from urllib.error import HTTPError
from urllib.parse import urlencode
from urllib.request import urlopen
from uuid import uuid4

import exchange_calendars as xcals
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf
from curl_cffi.requests import Session
from IPython.display import display
from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf

ROOT = next(
    (
        p
        for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (p / "research/experiment_registry.csv").is_file()
        and (p / "notebooks/volatility_targeting_2_research.ipynb").is_file()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError("Open Jupyter in the volatility-targeting experiment checkout.")
START = "2008-01-02"
END = "2026-10-02"
CORE = ("SPY", "TLT", "GLD")
MACRO = ("QQQ", "IWM", "HYG", "TLT", "GLD", "DBC")
TICKERS = ("SPY", "TLT", "GLD", "QQQ", "IWM", "HYG", "DBC")
REGISTRY_FIELDS = [
    "trial_id",
    "timestamp_utc",
    "hypothesis_hash",
    "code_commit",
    "data_hash",
    "family",
    "parameters",
    "sample",
    "transaction_cost_model",
    "status",
    "result_path",
    "conclusion",
    "oos_accessed",
]
WARNING = "WARNING: THE OFFICIAL OOS PERIOD HAS ALREADY BEEN ACCESSED. DO NOT RETUNE THE STRATEGY."


def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def git_commit(root):
    return subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=root, text=True
    ).strip()


def code_hash(root):
    document = json.loads(
        (root / "notebooks/volatility_targeting_2_research.ipynb").read_text()
    )
    cell_sources = [
        "".join(c["source"]) for c in document["cells"] if c["cell_type"] == "code"
    ]
    digest = hashlib.sha256(json.dumps(cell_sources, sort_keys=True).encode())
    digest.update((root / "pyproject.toml").read_bytes())
    digest.update((root / "uv.lock").read_bytes())
    return digest.hexdigest()


@dataclass(frozen=True)
class Variant:
    trial_id: str
    family: str
    target: float = 0.10
    base: str = "equal"
    universe: tuple = CORE
    lookback: int = 63
    threshold: float = 0.05
    cap: float = 1.0
    control: str = "event"
    covariance_days: int = 126
    har_train_min: int = 504

    def parameters(self):
        return asdict(self)


def variants():
    return [
        Variant("E0", "trailing", 0.08, control="monthly"),
        *[Variant(f"E1-{int(t * 100)}", "ensemble", t) for t in (0.09, 0.10, 0.11)],
        Variant("E2", "har"),
        Variant("E3-20", "range", lookback=20),
        Variant("E3-63", "range", lookback=63),
        Variant("E4", "ensemble", base="minvar"),
        Variant("E5", "ensemble", base="erc"),
        Variant("E6", "ensemble", base="erc", universe=MACRO),
        Variant("E7", "ensemble", control="asymmetric"),
        Variant("E8", "ensemble", cap=1.25, control="asymmetric"),
    ]


def assert_registration(root):
    spec = root / "research/HYPOTHESIS.md"
    registry = root / "research/experiment_registry.csv"
    if not spec.exists() or not registry.exists():
        raise RuntimeError("Missing committed hypothesis or registry")
    registered = pd.read_csv(registry)
    planned = {v.trial_id: v for v in variants()}
    original = registered[registered.status == "pre_registered"]
    if len(original) != 12 or set(original.trial_id) != set(planned):
        raise ValueError("Registry differs from the 12 pre-registered variants")
    for row in original.itertuples():
        if json.loads(row.parameters) != json.loads(
            json.dumps(planned[row.trial_id].parameters())
        ):
            raise ValueError("Registered parameters changed")
    # Files must be committed exactly before backtesting is possible.
    for path in (spec, registry):
        committed = subprocess.check_output(
            ["git", "show", f"HEAD:{path.relative_to(root)}"], cwd=root
        )
        if path == spec and committed != path.read_bytes():
            raise ValueError("Commit hypothesis before evaluating")
        if path == registry:
            committed_rows = [
                row
                for row in csv.DictReader(__import__("io").StringIO(committed.decode()))
                if row["status"] == "pre_registered"
            ]
            with registry.open(newline="") as handle:
                current_rows = [
                    row
                    for row in csv.DictReader(handle)
                    if row["status"] == "pre_registered"
                ]
            if current_rows != committed_rows:
                raise ValueError("Commit pre-registration before evaluating")
    print("Hypothesis SHA-256:", sha(spec))
    print("Registry SHA-256:", sha(registry))
    return sha(spec)


HYPOTHESIS_HASH = assert_registration(ROOT)
display(
    pd.DataFrame([v.parameters() for v in variants()])[
        ["trial_id", "family", "base", "target", "lookback", "control", "cap"]
    ]
)

## 2. Configuration
Research is the default. Final mode requires an external expected freeze hash. Freeze is opt-in and requires the exact current run ID after reviewing its evidence. Never place credentials in cells.

In [ ]:
CONFIG = {
    "data_start": "2008-01-02",
    "data_end": "2026-10-02",
    "mode": "final" if os.environ.get("GQH_FINAL_EVAL") == "1" else "research",
    "initial_nav": 1000000,
    "cost_bps": 5.0,
    "stress_cost_bps": 10.0,
    "risk_free_series": "DGS3MO",
    "random_seed": 42,
    "max_primary_variants": 12,
    "bootstrap_samples": 1000,
    "n_jobs": 1,
}
assert CONFIG["data_start"] == "2008-01-02" and CONFIG["data_end"] == "2026-10-02"
assert (
    CONFIG["cost_bps"] == 5
    and CONFIG["stress_cost_bps"] == 10
    and (CONFIG["random_seed"] == 42)
)
assert CONFIG["max_primary_variants"] == 12 and CONFIG["n_jobs"] == 1
MODE = CONFIG["mode"]
SELECTED_TRIAL = None
REVIEWED_RUN_ID = None
CONFIG

## 3. Imports and reproducibility
Runtime, packages, code revision, code hash and seed are saved per run. Sequential evaluation preserves a single append-only ledger; no unnecessary parallelism.

In [ ]:
np.random.seed(CONFIG["random_seed"])
RUN_ID = datetime.now(UTC).strftime("%Y%m%dT%H%M%S") + "-" + uuid4().hex[:8]
RUN_DIR = ROOT / "results" / RUN_ID
(RUN_DIR / "tables").mkdir(parents=True)
(RUN_DIR / "figures").mkdir()
OOS_FLAG = ROOT / "research/oos_accessed.flag"
STRATEGY_RESULTS, BASE_RESULTS, TRIAL_RUNS = {}, {}, []
if OOS_FLAG.exists():
    print(WARNING)
    if MODE == "research":
        raise RuntimeError("Research locked after OOS access")
runtime = {
    "python": sys.version,
    "platform": platform.platform(),
    "code_commit": git_commit(ROOT),
    "code_hash": code_hash(ROOT),
    "seed": 42,
    "mode": MODE,
    "packages": {
        name: importlib.metadata.version(name)
        for name in [
            "numpy",
            "pandas",
            "scipy",
            "scikit-learn",
            "yfinance",
            "exchange-calendars",
        ]
    },
}
(RUN_DIR / "runtime.json").write_text(json.dumps(runtime, indent=2))
(RUN_DIR / "resolved_config.json").write_text(json.dumps(CONFIG, indent=2))
print("Run:", RUN_ID, "—", MODE)
display(pd.Series(runtime["packages"], name="version").to_frame())
plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.size": 11,
        "figure.figsize": (11, 4.5),
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titleweight": "bold",
        "axes.grid": True,
        "grid.alpha": 0.15,
        "axes.prop_cycle": plt.cycler(
            color=["#2463A6", "#D07835", "#36826B", "#9271B1", "#B84B5F", "#677889"]
        ),
    }
)


def save_table(name, frame):
    frame.to_csv(RUN_DIR / "tables" / f"{name}.csv", index=False)
    return frame


def score_sample(result, name):
    if name == "validation":
        return result.loc["2019-01-01":]
    if name == "development":
        return result.iloc[551:].loc[:"2018-12-31"]
    if name == "OOS":
        return result.loc[boundary["oos_start"] :]
    return result.iloc[551:]


def figure(name, data, ylabel, title=None):
    data.to_csv(RUN_DIR / "figures" / f"{name}-data.csv")
    ax = data.plot(figsize=(11, 5), grid=True)
    ax.set_title(title or name.replace("_", " ").capitalize())
    ax.set_ylabel(ylabel)
    ax.set_xlabel("Session / scenario")
    ax.figure.tight_layout()
    ax.figure.savefig(RUN_DIR / "figures" / f"{name}.png", dpi=160)
    plt.show()
    plt.close(ax.figure)

## 4. Compute OOS split from calendar
The calendar boundary is computed before provider calls. The initial 550 sessions are common warm-up, not scored history.

In [ ]:
def calendar():
    return xcals.get_calendar("XNYS", start="2007-12-01", end="2026-10-31")


def split():
    sessions = calendar().sessions_in_range(START, END).tz_localize(None)
    twenty = sessions[-int(np.ceil(0.2 * len(sessions)))]
    two_year = sessions[sessions >= pd.Timestamp(END) - pd.DateOffset(years=2)][0]
    boundary = max(twenty, two_year)
    return {
        "sessions": sessions,
        "twenty_percent_start": twenty,
        "two_year_start": two_year,
        "oos_start": boundary,
        "binding_rule": "two years" if two_year >= twenty else "20%",
    }


boundary = split()
print("Total planned sessions:", len(boundary["sessions"]))
for key in ("twenty_percent_start", "two_year_start", "binding_rule", "oos_start"):
    print(key, boundary[key])
fig, ax = plt.subplots(figsize=(11, 2))
ax.axvspan(
    pd.Timestamp(CONFIG["data_start"]),
    boundary["oos_start"],
    color="steelblue",
    alpha=0.4,
    label="IS: development / validation",
)
ax.axvspan(
    boundary["oos_start"],
    pd.Timestamp(CONFIG["data_end"]),
    color="grey",
    alpha=0.4,
    label="LOCKED OOS",
)
ax.axvline(
    pd.Timestamp("2019-01-01"), color="black", linestyle="--", label="Validation start"
)
ax.legend(loc="upper left")
ax.set_yticks([])
ax.set_title("Planned chronological split; no prices used")
fig.tight_layout()
fig.savefig(RUN_DIR / "figures/oos_split.png", dpi=160)
plt.show()

## 5. Acquire in-sample data only
Yahoo raw OHLC, adjusted close, volume and actions; ALFRED DGS3MO initial releases. Missing FRED_API_KEY stops acquisition. Final mode skips this cell and opens data only after the gate.

In [ ]:
def guard_dates(index, end):
    if len(index) == 0 or pd.Timestamp(index.max()) >= pd.Timestamp(end):
        raise ValueError("Empty data or forbidden boundary/OOS observation")


def get_json(parameters):
    # Do not include request URLs in exceptions: they contain the API key.
    for attempt in range(4):
        try:
            with urlopen(
                "https://api.stlouisfed.org/fred/series/observations?"
                + urlencode(parameters),
                timeout=60,
            ) as response:
                return json.load(response)
        except HTTPError as error:
            if error.code not in (429, 500, 502, 503, 504):
                try:
                    reason = json.loads(error.read()).get(
                        "error_message", "Request rejected"
                    )
                except (ValueError, OSError):
                    reason = "Request rejected"
                reason = str(reason).replace(
                    str(parameters.get("api_key", "")), "[redacted]"
                )
                raise RuntimeError(
                    f"FRED request rejected (HTTP {error.code}): {reason}"
                ) from None
        except (OSError, ValueError):
            pass
        time.sleep(2**attempt)
    raise RuntimeError("FRED acquisition unavailable after bounded retries")


class BoundedYahooSession(Session):
    """Bound even yfinance's hidden latest-price timezone-discovery request."""

    def __init__(self, start, end):
        super().__init__(impersonate="chrome")
        self.start_epoch = int(pd.Timestamp(start, tz="America/New_York").timestamp())
        self.end_epoch = int(pd.Timestamp(end, tz="America/New_York").timestamp())
        self.chart_requests = []

    def bounded_parameters(self, url, params):
        params = dict(params or {})
        if "/finance/chart/" in url:
            if "range" in params:
                params.pop("range")
                params.update(period1=self.start_epoch, period2=self.end_epoch)
            if not {"period1", "period2"}.issubset(params):
                raise ValueError("Unbounded Yahoo price request blocked")
            if (
                int(params["period1"]) < self.start_epoch
                or int(params["period2"]) > self.end_epoch
            ):
                raise ValueError("Yahoo price request outside authorised date interval")
            self.chart_requests.append(
                {"url": url, "period1": params["period1"], "period2": params["period2"]}
            )
        elif "/finance/quote" in url:
            raise ValueError(
                "Current Yahoo quote request blocked during bounded acquisition"
            )
        return params

    def get(self, url, **kwargs):
        kwargs["params"] = self.bounded_parameters(url, kwargs.get("params"))
        return super().get(url, **kwargs)

In [ ]:
def acquire(end, cache, tickers=TICKERS, start=START):
    """Exclusive end bounds both ETF requests and ALFRED real-time horizon."""
    end = pd.Timestamp(end)
    cache = Path(cache)
    cache.mkdir(parents=True, exist_ok=True)
    prefix = cache / f"market-{pd.Timestamp(start):%Y%m%d}-{end:%Y%m%d}"
    metadata_path = prefix.with_suffix(".json")
    panel_path = prefix.with_suffix(".parquet")
    rates_path = cache / f"rates-{pd.Timestamp(start):%Y%m%d}-{end:%Y%m%d}.parquet"
    if metadata_path.exists():
        meta = json.loads(metadata_path.read_text())
        if meta.get("bounded_provider_policy") != 1:
            raise ValueError(
                "Cache predates bounded provider policy; use a fresh cache directory"
            )
        if (
            sha(panel_path) != meta["market_sha256"]
            or sha(rates_path) != meta["rates_sha256"]
        ):
            raise ValueError("Cached data hash mismatch")
        panel, rates = pd.read_parquet(panel_path), pd.read_parquet(rates_path)
        guard_dates(panel.index, end)
        guard_dates(pd.DatetimeIndex(rates.date), end)
        if pd.to_datetime(rates.realtime_start).max() >= end:
            raise ValueError("Cached rates include forbidden release dates")
        return panel, rates, meta
    key = os.environ.get("FRED_API_KEY")
    if not key:
        raise RuntimeError(
            "Set FRED_API_KEY in ignored .env for ALFRED initial-release DGS3MO; revised-rate fallback is prohibited"
        )
    observations = []
    observation_start = pd.Timestamp(start) - pd.Timedelta(days=32)
    last_allowed = end - pd.Timedelta(days=1)
    for year in range(observation_start.year, last_allowed.year + 1):
        first = max(observation_start, pd.Timestamp(year=year, month=1, day=1))
        last = min(last_allowed, pd.Timestamp(year=year, month=12, day=31))
        release_end = min(last_allowed, last + pd.Timedelta(days=32))
        offset = 0
        print(f"ALFRED initial releases: {first.date()} to {last.date()}", flush=True)
        while True:
            payload = get_json(
                {
                    "series_id": "DGS3MO",
                    "api_key": key,
                    "file_type": "json",
                    "observation_start": str(first.date()),
                    "observation_end": str(last.date()),
                    "realtime_start": str(first.date()),
                    "realtime_end": str(release_end.date()),
                    "output_type": 4,
                    "limit": 10000,
                    "offset": offset,
                }
            )
            page = payload.get("observations", [])
            observations.extend(page)
            offset += len(page)
            if offset >= int(payload["count"]):
                break
            if not page:
                raise RuntimeError("FRED pagination stopped before reported count")
    rates = pd.DataFrame(observations)
    if rates.empty:
        raise RuntimeError("No ALFRED initial-release observations")
    rates["date"] = pd.to_datetime(rates.date)
    rates["realtime_start"] = pd.to_datetime(rates.realtime_start)
    rates["value"] = pd.to_numeric(rates.value, errors="coerce")
    rates = rates.dropna(subset=["value"])
    guard_dates(pd.DatetimeIndex(rates.date), end)
    guard_dates(pd.DatetimeIndex(rates.realtime_start), end)
    panels = {}
    session = BoundedYahooSession(start, str(end.date()))
    for ticker in tickers:
        print(f"Acquiring {ticker}: {START} <= date < {end.date()}", flush=True)
        frame = None
        for attempt in range(3):
            try:
                frame = yf.Ticker(ticker, session=session).history(
                    start=start,
                    end=str(end.date()),
                    auto_adjust=False,
                    actions=True,
                    repair=False,
                    raise_errors=True,
                )
                if not frame.empty:
                    break
            except Exception:  # noqa: BLE001 - vendor exception types vary; never expose request details
                frame = None
            time.sleep(2**attempt)
        if frame is None or frame.empty:
            raise RuntimeError(f"Real market data unavailable for {ticker}")
        frame.index = frame.index.tz_localize(None).normalize()
        guard_dates(frame.index, end)
        panels[ticker] = frame
    panel = pd.concat(panels, axis=1)
    panel.to_parquet(panel_path)
    rates.to_parquet(rates_path, index=False)
    meta = {
        "source": "Yahoo Finance via yfinance; ALFRED DGS3MO initial release (output_type=4)",
        "retrieved_utc": datetime.now(UTC).isoformat(),
        "request_start": start,
        "bounded_chart_requests": session.chart_requests,
        "bounded_provider_policy": 1,
        "request_end_exclusive": str(end.date()),
        "adjustment": "Yahoo Adj Close; raw OHLC and corporate actions retained; repair=False",
        "market_sha256": sha(panel_path),
        "rates_sha256": sha(rates_path),
        "paths": [str(panel_path), str(rates_path)],
        "tickers": list(tickers),
    }
    metadata_path.write_text(json.dumps(meta, indent=2))
    return panel, rates, meta

In [ ]:
def prepare(panel, rates, end):
    planned = split()["sessions"]
    expected = planned[planned < pd.Timestamp(end)]
    guard_dates(panel.index, end)
    audits = []
    for ticker in panel.columns.get_level_values(0).unique():
        frame = panel[ticker]
        valid = frame["Adj Close"].dropna()
        audits.append(
            {
                "ticker": ticker,
                "first_date": valid.index.min(),
                "last_date": valid.index.max(),
                "row_count": len(valid),
                "duplicate_dates": frame.index.duplicated().sum(),
                "missing_sessions": len(expected.difference(valid.index)),
                "extra_sessions": len(frame.index.difference(expected)),
                "missing_adjusted_close": frame["Adj Close"].isna().sum(),
                "non_positive_prices": (
                    frame[["Open", "High", "Low", "Close", "Adj Close"]] <= 0
                )
                .sum()
                .sum(),
                "invalid_ranges": (
                    (frame.High < frame.Low)
                    | (frame.High < frame.Close)
                    | (frame.Low > frame.Close)
                ).sum(),
                "invalid_volume": ((frame.Volume < 0) | frame.Volume.isna()).sum(),
                "suspicious_returns": (
                    valid.pct_change(fill_method=None).abs() > 0.30
                ).sum(),
                "dividend_events": int((frame.get("Dividends", 0) != 0).sum()),
                "split_events": int((frame.get("Stock Splits", 0) != 0).sum()),
            }
        )
    audit = pd.DataFrame(audits)
    error_columns = [
        "duplicate_dates",
        "missing_sessions",
        "extra_sessions",
        "missing_adjusted_close",
        "non_positive_prices",
        "invalid_ranges",
        "invalid_volume",
    ]
    if audit[error_columns].to_numpy().sum():
        raise ValueError(
            "Structural market data errors:\n" + audit.to_string(index=False)
        )
    if audit.suspicious_returns.sum():
        raise ValueError(
            "Returns over 30% require documented investigation before evaluation"
        )
    price = panel.xs("Adj Close", axis=1, level=1).reindex(expected)
    # Availability is strictly AFTER release date, even if released before close.
    releases = rates.sort_values(["realtime_start", "date"]).copy()
    releases["available"] = [
        planned[planned > max(d, r)][0] if (planned > max(d, r)).any() else pd.NaT
        for d, r in zip(releases.date, releases.realtime_start)
    ]
    releases = releases.dropna(subset=["available"]).groupby("available").tail(1)
    daily = pd.DataFrame({"date": expected})
    merged = pd.merge_asof(
        daily, releases.sort_values("available"), left_on="date", right_on="available"
    )
    if merged.value.isna().any():
        raise ValueError("No available initial-release cash rate for early sessions")
    elapsed = expected.to_series().diff().dt.days.fillna(1).to_numpy()
    # Return over interval ending t uses rate known at previous close.
    yield_rate = pd.Series(merged.value.to_numpy() / 100, index=expected)
    rf = (1 + yield_rate.shift(1).fillna(yield_rate.iloc[0])) ** (elapsed / 365) - 1
    closes = pd.Series([calendar().session_close(d) for d in expected], index=expected)
    return price, rf, closes, audit


def load_market_data(final=False):
    global \
        DATA_METADATA, \
        RAW_CLOSE, \
        cash_returns, \
        closes, \
        highs, \
        lows, \
        panel, \
        prices, \
        volumes
    if final and MODE != "final":
        raise RuntimeError("Final acquisition forbidden in research mode")
    if final:
        if not globals().get("FROZEN") or not OOS_FLAG.exists():
            raise RuntimeError("Acquire holdout only after exclusive final gate")
        end = pd.Timestamp(END) + pd.Timedelta(days=1)
    else:
        end = boundary["oos_start"]
    panel, rates, DATA_METADATA = acquire(
        end,
        ROOT / "data/cache/volatility-targeting-2",
        start=str(boundary["oos_start"].date()) if final else START,
    )
    if final:
        old_panel = pd.read_parquet(ROOT / FROZEN["research_data"]["paths"][0])
        old_rates = pd.read_parquet(ROOT / FROZEN["research_data"]["paths"][1])
        panel = pd.concat([old_panel, panel])
        rates = pd.concat([old_rates, rates]).drop_duplicates(
            ["date", "realtime_start"]
        )
        combined_path = RUN_DIR / "combined-market.parquet"
        combined_rates = RUN_DIR / "combined-rates.parquet"
        panel.to_parquet(combined_path)
        rates.to_parquet(combined_rates)
        DATA_METADATA["market_sha256"] = sha(combined_path)
        DATA_METADATA["rates_sha256"] = sha(combined_rates)
        DATA_METADATA["frozen_is_provenance"] = FROZEN["research_data"]
    prices, cash_returns, closes, audit = prepare(panel, rates, end)
    highs = panel.xs("High", axis=1, level=1)
    lows = panel.xs("Low", axis=1, level=1)
    volumes = panel.xs("Volume", axis=1, level=1)
    RAW_CLOSE = panel.xs("Close", axis=1, level=1)
    if not final:
        assert prices.index.max() < boundary["oos_start"]
    for key in ["retrieved_utc", "source", "market_sha256", "rates_sha256"]:
        audit[key] = DATA_METADATA[key]
    save_table("data_audit", audit)
    (RUN_DIR / "data_provenance.json").write_text(json.dumps(DATA_METADATA, indent=2))
    print(
        "Real observed sessions loaded:",
        len(prices),
        prices.index.min(),
        prices.index.max(),
    )
    return audit

In [ ]:
audit = None
if MODE == "research":
    audit = load_market_data()
    assert prices.index.max() < boundary["oos_start"]
    returns = prices.pct_change(fill_method=None)
    print(f"{len(prices):,} real market sessions; OOS prices absent")

## 6. Data audit
Missing sessions, prices, non-positive values, invalid ranges, duplicates or suspicious returns stop evaluation. No price forward-filling. Data hashes and retrieval metadata accompany tables.

In [ ]:
if MODE == "research":
    if audit is None:
        raise RuntimeError("Run acquisition successfully before inspecting the audit.")
    display(audit)
    print((RUN_DIR / "data_provenance.json").read_text())

## 7. Returns and risk-free processing
Adjusted-close simple daily returns; first-release rate available strictly after its release date, then lagged over each accrual interval. ACT/365 cash proxy; 252-session performance annualisation. Undefined metrics remain NaN.

In [ ]:
if MODE == "research":
    display(
        returns.describe().T.assign(skew=returns.skew(), excess_kurtosis=returns.kurt())
    )
    figure(
        "asset_histories",
        prices / prices.iloc[0],
        "Adjusted-close wealth per initial dollar",
    )
    figure(
        "asset_rolling_vol",
        returns.rolling(63).std() * np.sqrt(252),
        "Annualised 63-session volatility",
    )
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ticker in ["SPY", "TLT", "GLD"]:
        axes[0].hist(
            returns[ticker].dropna(),
            bins=60,
            density=True,
            histtype="step",
            label=ticker,
        )
    axes[0].set(
        xlabel="Daily total return", ylabel="Density", title="Return distributions"
    )
    axes[0].legend()
    correlation = returns[["SPY", "TLT", "GLD"]].corr()
    image = axes[1].imshow(correlation, vmin=-1, vmax=1, cmap="RdBu_r")
    axes[1].set(
        xticks=range(3),
        yticks=range(3),
        xticklabels=correlation.columns,
        yticklabels=correlation.columns,
        title="IS correlations: descriptive",
    )
    fig.colorbar(image, ax=axes[1])
    fig.tight_layout()
    fig.savefig(RUN_DIR / "figures/returns_and_correlations.png", dpi=160)
    plt.show()
    returns.to_csv(RUN_DIR / "daily_asset_returns.csv")

## 8. Baseline replication
E0 is evaluated first by the unified runner below: monthly equal-weight SPY/TLT/GLD, 63-session sample volatility, 8% target. Its cost and fill conventions may differ from the prior study; this is an explicit contract replication, not an assertion of identical prior results.

In [ ]:
if MODE == "research":
    baseline_returns = returns[["SPY", "TLT", "GLD"]].mean(axis=1)
    baseline_volatility = baseline_returns.rolling(63).std() * np.sqrt(252)
    baseline_raw_exposure = (0.08 / baseline_volatility).clip(0, 1)
    figure(
        "baseline_volatility_preview",
        baseline_volatility.to_frame("63-day forecast"),
        "Annualised volatility",
    )
    figure(
        "baseline_exposure_preview",
        baseline_raw_exposure.to_frame("raw target"),
        "Risky NAV fraction; monthly execution follows in section 12",
    )

## 9. Forecast engines
Trailing sample volatility; maximum 20/63-span EWMA variance; expanding monthly-refit HAR with matured next-21-session labels; Parkinson single-window 20/63 with trailing close-return correlation. The estimator code is directly below. Range estimates omit overnight risk.

### Construct and compare the volatility forecasts
The following functions are used by the simulator below. EWMA uses squared returns; HAR labels span the next 21 sessions and must mature before fitting. Inspect the forecasts before evaluating a strategy. Future realised volatility is a diagnostic target, never a signal input.

In [ ]:
def har_prediction(returns, position, state, minimum=504):
    """Labels j+1:j+21 must have fully matured by position; monthly refit."""
    squared = returns.pow(2)
    features = pd.concat(
        [squared, squared.rolling(5).mean(), squared.rolling(21).mean()], axis=1
    )
    target = squared.iloc[::-1].rolling(21).mean().iloc[::-1].shift(-1)
    train = (
        pd.concat([features, target.rename("target")], axis=1)
        .iloc[: position - 20]
        .dropna()
    )
    if len(train) < minimum:
        return np.nan
    month = returns.index[position].to_period("M")
    if state.get("month") != month:
        x = np.column_stack([np.ones(len(train)), train.iloc[:, :3]])
        state["beta"] = np.linalg.lstsq(x, train["target"], rcond=None)[0]
        state["month"] = month
    value = float(np.r_[1.0, features.iloc[position].to_numpy()] @ state["beta"])
    return np.sqrt(252 * max(value, 1e-08))


def forecast(history, base, variant, high=None, low=None, state=None):
    portfolio = history @ base
    if variant.family == "trailing":
        return float(portfolio.tail(variant.lookback).std(ddof=1) * np.sqrt(252))
    if variant.family == "ensemble":
        fast = (
            portfolio.pow(2).ewm(span=20, adjust=False, min_periods=63).mean().iloc[-1]
        )
        slow = (
            portfolio.pow(2)
            .ewm(span=variant.lookback, adjust=False, min_periods=63)
            .mean()
            .iloc[-1]
        )
        return float(np.sqrt(252 * max(fast, slow)))
    if variant.family == "har":
        return har_prediction(
            portfolio, len(portfolio) - 1, state, variant.har_train_min
        )
    variance = np.log(high / low).pow(2).tail(variant.lookback).mean() / (4 * np.log(2))
    correlation = history.tail(variant.lookback).corr().fillna(0).to_numpy(copy=True)
    np.fill_diagonal(correlation, 1)
    covariance = np.outer(np.sqrt(variance), np.sqrt(variance)) * correlation
    return float(np.sqrt(max(252 * base @ covariance @ base, 1e-12)))

In [ ]:
if MODE == "research":
    equal_base_returns = returns[["SPY", "TLT", "GLD"]].mean(axis=1)
    squared = equal_base_returns.pow(2)
    volatility_forecasts = pd.DataFrame(
        {
            "trailing_63": equal_base_returns.rolling(63).std() * np.sqrt(252),
            "EWMA_fast_20": np.sqrt(
                252 * squared.ewm(span=20, adjust=False, min_periods=63).mean()
            ),
            "EWMA_slow_63": np.sqrt(
                252 * squared.ewm(span=63, adjust=False, min_periods=63).mean()
            ),
        }
    )
    volatility_forecasts["ensemble"] = volatility_forecasts[
        ["EWMA_fast_20", "EWMA_slow_63"]
    ].max(axis=1)
    figure("forecast_engines", volatility_forecasts, "Annualised forecast volatility")
    har_features = pd.DataFrame(
        {
            "RV1": squared,
            "RV5": squared.rolling(5).mean(),
            "RV21": squared.rolling(21).mean(),
        }
    )
    display(har_features.dropna().head())

## 10. Base portfolio engines
Monthly equal-weight, bounded Ledoit-Wolf minimum variance and constrained ERC. Covariance window 126; three-asset cap 70%, six-asset cap 35% of base. No full-sample covariance fit.

In [ ]:
def allocation(history, kind, cap):
    n = history.shape[1]
    if kind == "equal":
        return np.full(n, 1 / n)
    covariance = LedoitWolf().fit(history.to_numpy()).covariance_
    scale = max(np.trace(covariance), 1e-12)
    covariance = covariance / scale

    def objective(w):
        if kind == "minvar":
            return float(w @ covariance @ w)
        contribution = w * (covariance @ w)
        return float(np.sum((contribution - contribution.mean()) ** 2))

    fit = minimize(
        objective,
        np.full(n, 1 / n),
        method="SLSQP",
        bounds=[(0, cap)] * n,
        constraints={"type": "eq", "fun": lambda w: w.sum() - 1},
        options={"ftol": 1e-12, "maxiter": 1000},
    )
    if not fit.success or abs(fit.x.sum() - 1) > 1e-07 or (fit.x > cap + 1e-07).any():
        raise RuntimeError(f"Unsupported {kind} allocation: {fit.message}")
    return fit.x

In [ ]:
if MODE == "research":
    development_returns = (
        returns.loc[:"2018-12-31", ["SPY", "TLT", "GLD"]].tail(126).dropna()
    )
    allocation_example = pd.DataFrame(
        {
            kind: allocation(development_returns, kind, 0.7)
            for kind in ["equal", "minvar", "erc"]
        },
        index=development_returns.columns,
    )
    display(allocation_example)
    covariance_example = LedoitWolf().fit(development_returns).covariance_
    risk_contributions = {}
    for kind in allocation_example:
        w = allocation_example[kind].to_numpy()
        risk_contributions[kind] = (
            w * (covariance_example @ w) / (w @ covariance_example @ w)
        )
    display(
        pd.DataFrame(risk_contributions, index=allocation_example.index).rename_axis(
            "asset"
        )
    )
    print("Example uses development information through 2018-12-31; no future fit.")

## 11. Exposure control engines
Monthly baseline; 5 pp event-driven trigger; asymmetric restoration after five consecutive qualifying sessions. E8 leverage constraints override hysteresis. Drift can move realised exposure beyond an order limit between closes; this is reported rather than silently rebalanced.

In [ ]:
def control(raw, current, variant, streak):
    if variant.control == "monthly":
        return (raw, 0)
    if variant.control == "event":
        return (raw if abs(raw - current) >= variant.threshold else current, 0)
    if raw - current <= -variant.threshold:
        return (raw, 0)
    streak = streak + 1 if raw - current >= 0.1 - 1e-12 else 0
    if streak >= 5:
        return (min(raw, current + 0.1), 0)
    return (current, streak)

## 12. Unified causal simulator
One simulator handles every family and comparator. Pending orders earn the old holdings’ return through the execution close. Fees use drifted holdings and solve post-fee NAV accounting. Orders save feature, signal, execution and first affected return timestamps. Same-bar fills are asserted against.

In [ ]:
def simulate(
    prices,
    rf,
    closes,
    variant,
    high=None,
    low=None,
    cost_bps=5.0,
    borrow_bps=50.0,
    overlay=True,
    static_exposure=None,
    start_position=504,
    forecast_engine=None,
    allocation_engine=None,
    control_engine=None,
):
    """Close orders execute one session after formation, after that day's return.

    Weights are post-cost NAV fractions. Costs are solved self-consistently
    against drifted dollar holdings, so the cash account and fees reconcile.
    """
    forecast_engine = forecast_engine or forecast
    allocation_engine = allocation_engine or allocation
    control_engine = control_engine or control
    prices = prices.loc[:, list(variant.universe)]
    if (
        prices.isna().any().any()
        or not prices.index.is_monotonic_increasing
        or prices.index.has_duplicates
    ):
        raise ValueError(
            "Invalid price history: missing, duplicate or unordered observations"
        )
    returns = prices.pct_change(fill_method=None)
    returns.iloc[0] = 0.0
    rf = rf.reindex(prices.index)
    if rf.isna().any() or (prices <= 0).any().any():
        raise ValueError("Missing available risk-free rate or invalid prices")
    n = prices.shape[1]
    holdings = np.zeros(n)
    cash = 1.0
    base = np.full(n, 1 / n)
    pending = None
    exposure = 0.0
    streak = 0
    har_state = {}
    correlations = []
    if variant.cap > 1:
        for j in range(20, start_position):
            matrix = returns.iloc[j - 19 : j + 1].corr().to_numpy()
            correlations.append(float(matrix[np.triu_indices(n, 1)].mean()))
    rows, orders = [], []
    weight_rows, base_rows, trade_rows, forecast_weight_rows = [], [], [], []
    previous_month = None
    for t, date in enumerate(prices.index):
        old_nav = holdings.sum() + cash
        risky = float(holdings @ returns.iloc[t].to_numpy())
        cash_pnl = cash * rf.iloc[t] if cash >= 0 else 0.0
        financing = (
            -cash
            * (
                rf.iloc[t]
                + borrow_bps / 10000 / 365 * (date - prices.index[t - 1]).days
            )
            if cash < 0 and t
            else 0.0
        )
        holdings *= 1 + returns.iloc[t].to_numpy()
        cash += cash_pnl - financing
        before_fee = holdings.sum() + cash
        trade = np.zeros(n)
        fee = turnover = 0.0
        if pending is not None:
            desired, signal_date, signal_base, signal_exposure = pending
            # Solve fee = c * sum(abs(post_fee_nav*target - drifted_holdings)).
            after_fee = before_fee
            for _ in range(50):
                updated = (
                    before_fee
                    - cost_bps / 10000 * np.abs(after_fee * desired - holdings).sum()
                )
                if abs(updated - after_fee) < 1e-14:
                    break
                after_fee = updated
            after_fee = updated
            trade = after_fee * desired - holdings
            fee = before_fee - after_fee
            old_weights = np.r_[holdings / before_fee, cash / before_fee]
            holdings = after_fee * desired
            cash = after_fee - holdings.sum()
            new_weights = np.r_[desired, 1 - desired.sum()]
            turnover = 0.5 * np.abs(new_weights - old_weights).sum()
            exposure = signal_exposure
            base = signal_base
            effective = closes.iloc[t + 1] if t + 1 < len(prices) else pd.NaT
            assert closes.loc[signal_date] < closes.loc[date]
            orders.append(
                {
                    "signal_timestamp": closes.loc[signal_date],
                    "feature_timestamp": closes.loc[signal_date],
                    "execution_timestamp": closes.loc[date],
                    "effective_from_timestamp": closes.loc[date],
                    "first_return_end_timestamp": effective,
                    "exposure": exposure,
                }
            )
            pending = None
        nav = holdings.sum() + cash
        if nav <= 0:
            raise RuntimeError("Insolvent simulation")
        rows.append(
            {
                "net_return": nav / old_nav - 1,
                "gross_return": (risky + cash_pnl - financing) / old_nav,
                "risky_contribution": risky / old_nav,
                "cash_contribution": cash_pnl / old_nav,
                "financing_cost": financing / old_nav,
                "transaction_cost": fee / old_nav,
                "traded_notional": np.abs(trade).sum() / before_fee,
                "turnover": turnover,
                "equity": nav,
                "exposure": holdings.sum() / nav,
                "forecast": np.nan,
                "raw_exposure": np.nan,
            }
        )
        weight_rows.append(holdings / nav)
        base_rows.append(base.copy())
        forecast_weight_rows.append(base.copy())
        trade_rows.append(trade / before_fee)
        if t < start_position or t == len(prices) - 1:
            continue
        month = date.to_period("M")
        # Monthly signal on last actual NYSE session of month, not first session.
        month_end = prices.index[t + 1].to_period("M") != month
        first = previous_month is None
        base_change = first or month_end
        next_base = (
            allocation_engine(
                returns.iloc[max(1, t - variant.covariance_days + 1) : t + 1],
                variant.base,
                0.35 if n == 6 else 0.70,
            )
            if base_change
            else base
        )
        history = returns.iloc[1 : t + 1]
        # HAR is only registered for the constant equal-weight base.
        vol = forecast_engine(
            history,
            next_base,
            variant,
            None if high is None else high.loc[:date, prices.columns],
            None if low is None else low.loc[:date, prices.columns],
            har_state,
        )
        rows[-1]["forecast"] = vol
        forecast_weight_rows[-1] = next_base.copy()
        if not np.isfinite(vol) or vol <= 0:
            continue
        raw = min(variant.cap, variant.target / vol) if overlay else 1.0
        if static_exposure is not None:
            raw = static_exposure
        if variant.cap > 1 and overlay and static_exposure is None:
            p = history @ next_base
            wealth = (1 + p.tail(20)).cumprod()
            dd = wealth.iloc[-1] / max(1.0, wealth.max()) - 1
            corr = history.tail(20).corr().to_numpy()
            mean_corr = float(corr[np.triu_indices(n, 1)].mean())
            correlations.append(mean_corr)
            ceiling = np.quantile(correlations, 0.8)
            if vol > variant.target or dd < -0.05 or mean_corr > ceiling:
                raw = min(raw, 1.0)
        rows[-1]["raw_exposure"] = raw
        if variant.control == "monthly" and not (base_change or first):
            continue
        actual = holdings.sum() / nav
        desired_exposure, streak = (
            control_engine(raw, actual, variant, streak)
            if overlay and static_exposure is None
            else (raw, 0)
        )
        if first:
            desired_exposure, streak = raw, 0
        # Hard leverage ceiling overrides slow restoration/ordinary hysteresis.
        if variant.cap > 1 and raw <= 1 and actual > 1:
            desired_exposure = min(desired_exposure, 1.0)
        if first or base_change or abs(desired_exposure - actual) > 1e-10:
            desired_exposure = min(variant.cap, max(0.0, desired_exposure))
            pending = (next_base * desired_exposure, date, next_base, desired_exposure)
        previous_month = month
    result = pd.DataFrame(rows, index=prices.index)
    result.attrs["orders"] = pd.DataFrame(orders)
    result.attrs["weights"] = pd.DataFrame(
        weight_rows, index=prices.index, columns=prices.columns
    )
    result.attrs["base_weights"] = pd.DataFrame(
        base_rows, index=prices.index, columns=prices.columns
    )
    result.attrs["forecast_weights"] = pd.DataFrame(
        forecast_weight_rows, index=prices.index, columns=prices.columns
    )
    result.attrs["trades"] = pd.DataFrame(
        trade_rows, index=prices.index, columns=prices.columns
    )
    return result


def metrics(result, rf):
    r = result.net_return
    if len(r) < 2:
        raise ValueError("Insufficient metric observations")
    excess = r - rf.reindex(r.index)
    wealth = pd.concat(
        [pd.Series([1.0]), (1 + r).cumprod().reset_index(drop=True)], ignore_index=True
    )
    dd = wealth / wealth.cummax() - 1
    cagr = float(wealth.iloc[-1] ** (252 / len(r)) - 1)
    vol = float(r.std(ddof=1) * np.sqrt(252))
    downside = float(np.sqrt(np.mean(np.minimum(excess, 0) ** 2)) * np.sqrt(252))
    var = float(r.quantile(0.05))
    tail = r[r <= var]
    durations, duration = [], 0
    for value in dd:
        duration = duration + 1 if value < -1e-12 else 0
        durations.append(duration)
    return {
        "cagr": cagr,
        "volatility": vol,
        "sharpe": excess.mean() * 252 / vol if vol else np.nan,
        "sortino": excess.mean() * 252 / downside if downside else np.nan,
        "max_drawdown": float(dd.min()),
        "calmar": cagr / abs(dd.min()) if dd.min() < 0 else np.nan,
        "var_95": var,
        "cvar_95": float(tail.mean()),
        "worst_1d": float(r.min()),
        "worst_5d": float((1 + r).rolling(5).apply(np.prod, raw=True).min() - 1),
        "worst_20d": float((1 + r).rolling(20).apply(np.prod, raw=True).min() - 1),
        "turnover": result.turnover.sum() * 252 / len(r),
        "implementation_cost": result.transaction_cost.sum() * 252 / len(r),
        "terminal_wealth": float(wealth.iloc[-1]),
        "recovery_sessions": max(durations),
        "unrecovered_at_end": bool(dd.iloc[-1] < -1e-12),
    }


def passes(strategy, base):
    return (
        strategy["volatility"] <= 0.9 * base["volatility"]
        and abs(strategy["max_drawdown"]) <= 0.9 * abs(base["max_drawdown"])
        and strategy["cvar_95"] > base["cvar_95"]
        and strategy["cagr"] >= base["cagr"] - 0.015
        and strategy["turnover"] < 2.0
    )

In [ ]:
def append_row(path, row, fields):
    exists = path.exists()
    with path.open("a", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields, lineterminator="\n")
        if not exists:
            writer.writeheader()
        writer.writerow({field: row.get(field, "") for field in fields})
        handle.flush()
        os.fsync(handle.fileno())


def evaluate_variant(
    variant, cost=5.0, borrow=50.0, overlay=True, static=None, tag="primary"
):
    trial = f"{variant.trial_id}-{tag}-{uuid4().hex[:10]}"
    path = RUN_DIR / f"{trial}.csv"
    parameters = {
        **variant.parameters(),
        "overlay": overlay,
        "static_exposure": static,
        "borrow_bps": borrow,
        "diagnostic": tag,
    }
    row = {
        "trial_id": trial,
        "timestamp_utc": datetime.now(UTC).isoformat(),
        "hypothesis_hash": HYPOTHESIS_HASH,
        "code_commit": git_commit(ROOT),
        "data_hash": DATA_METADATA["market_sha256"]
        + ":"
        + DATA_METADATA["rates_sha256"],
        "family": variant.family,
        "parameters": json.dumps(parameters, sort_keys=True),
        "sample": "OOS" if MODE == "final" else "development+reused_validation",
        "transaction_cost_model": f"{cost} bp risky purchases+sales; borrow rf+{borrow} bp",
        "result_path": str(path.relative_to(ROOT)),
        "oos_accessed": OOS_FLAG.exists(),
    }

    def log(status, conclusion):
        entry = {**row, "status": status, "conclusion": conclusion}
        append_row(ROOT / "research/experiment_registry.csv", entry, REGISTRY_FIELDS)
        ledger = ROOT / "research/experiments.csv"
        fields = next(csv.reader(ledger.open()))
        append_row(
            ledger,
            {
                **entry,
                "hypothesis_commit": git_commit(ROOT),
                "data_sha256": row["data_hash"],
                "config_sha256": hashlib.sha256(row["parameters"].encode()).hexdigest(),
                "cost_model": row["transaction_cost_model"],
                "holdout_access": row["oos_accessed"],
            },
            fields,
        )

    log("started", "Attempt retained even if process fails")
    try:
        result = simulate(
            prices,
            cash_returns,
            closes,
            variant,
            highs,
            lows,
            cost,
            borrow,
            overlay,
            static,
            start_position=550,
        )
        result.to_csv(path)
        for name in ("orders", "weights", "base_weights", "forecast_weights", "trades"):
            result.attrs[name].to_csv(RUN_DIR / f"{trial}-{name}.csv")
        orders = result.attrs["orders"]
        if not orders.empty:
            assert (orders.feature_timestamp <= orders.signal_timestamp).all()
            assert (orders.signal_timestamp < orders.execution_timestamp).all()
        assert np.allclose(
            result.net_return,
            result.risky_contribution
            + result.cash_contribution
            - result.financing_cost
            - result.transaction_cost,
        )
    except Exception as error:
        log("failed", f"{type(error).__name__}: {error}")
        raise
    log(
        "completed",
        "No OOS retuning; diagnostic not eligible for selection"
        if tag != "primary"
        else "Await validation",
    )
    TRIAL_RUNS.append(row)
    return result

### Execute E0 and inspect the accounting
The simulator below implements the timeline explicitly. At the execution close, old holdings earn that session’s return; the new position affects the next interval. The same engine is used for every candidate.

In [ ]:
if MODE == "research":
    baseline = variants()[0]
    STRATEGY_RESULTS["E0"] = evaluate_variant(baseline)
    BASE_RESULTS["E0"] = evaluate_variant(baseline, overlay=False, tag="matched_base")
    e0 = score_sample(STRATEGY_RESULTS["E0"], "in_sample")
    e0_base = score_sample(BASE_RESULTS["E0"], "in_sample")
    display(
        pd.DataFrame(
            {
                "E0": metrics(e0, cash_returns),
                "unscaled": metrics(e0_base, cash_returns),
            }
        ).T
    )
    figure(
        "E0_equity",
        (
            1 + pd.DataFrame({"E0": e0.net_return, "unscaled": e0_base.net_return})
        ).cumprod(),
        "Net wealth; 5 bp per risky purchase/sale",
    )
    display(STRATEGY_RESULTS["E0"].attrs["orders"].head())

In [ ]:
if MODE == "research":
    display(
        pd.DataFrame([v.parameters() for v in variants()]).drop(columns=["universe"])
    )

## 13. Run the twelve registered primary candidates
Each candidate and matched base uses the same simulator, costs and scored dates. Failures are retained, with progress printed after each run. Validation, sensitivities and plots follow in separate stages; this cell does not freeze a winner.

In [ ]:
primary_failures = []
if MODE == "research":
    for number, variant in enumerate(variants()[1:], 2):
        print(
            f"{number}/12 — {variant.trial_id}: {variant.family}, {variant.base}",
            flush=True,
        )
        try:
            strategy = evaluate_variant(variant)
            matched_base = evaluate_variant(variant, overlay=False, tag="matched_base")
            STRATEGY_RESULTS[variant.trial_id] = strategy
            BASE_RESULTS[variant.trial_id] = matched_base
            display(
                pd.DataFrame(
                    {
                        "strategy": metrics(
                            score_sample(strategy, "validation"), cash_returns
                        ),
                        "matched_base": metrics(
                            score_sample(matched_base, "validation"), cash_returns
                        ),
                    }
                ).T
            )
        except Exception as error:  # noqa: BLE001 - retain each failed candidate
            primary_failures.append(
                {
                    "trial_id": variant.trial_id,
                    "passed": False,
                    "failure_reason": str(error),
                }
            )
            print("FAILED AND RETAINED:", variant.trial_id, error, flush=True)

## 14. Validation results
Apply all precommitted financial, forecast, yearly-fold, doubled-cost and neighbour gates. No OOS data are available.

In [ ]:
def forecast_diagnostics(variant, result):
    weights = result.attrs["forecast_weights"]
    r = prices[list(variant.universe)].pct_change(fill_method=None)
    future = pd.Series(np.nan, index=r.index)
    for t in range(len(r) - 21):
        future.iloc[t] = np.sqrt(
            252
            * np.mean(
                (r.iloc[t + 1 : t + 22].to_numpy() @ weights.iloc[t].to_numpy()) ** 2
            )
        )
    paired = pd.DataFrame({"forecast": result.forecast, "future": future}).dropna()
    rows = []
    for sample, subset in [
        ("development", paired.loc[:"2018-11-30"]),
        ("validation", paired.loc["2019-01-01":]),
    ]:
        difference = subset.forecast - subset.future
        rows.append(
            {
                "trial_id": variant.trial_id,
                "sample": sample,
                "observations": len(subset),
                "mae": difference.abs().mean(),
                "rmse": np.sqrt(np.mean(difference**2)),
                "pearson_ic": subset.forecast.corr(subset.future),
                "rank_ic": subset.forecast.corr(subset.future, method="spearman"),
            }
        )
    paired.to_csv(RUN_DIR / f"{variant.trial_id}-forecast-pairs.csv")
    return pd.DataFrame(rows)


def neighbour_diagnostics(variant):
    settings = [
        ("target_low", replace(variant, target=variant.target - 0.01)),
        ("target_high", replace(variant, target=variant.target + 0.01)),
        (
            "lookback_low",
            replace(
                variant,
                lookback=round(variant.lookback * 0.8),
                har_train_min=403,
                covariance_days=101,
            ),
        ),
        (
            "lookback_high",
            replace(
                variant,
                lookback=round(variant.lookback * 1.2),
                har_train_min=605,
                covariance_days=151,
            ),
        ),
        ("trigger_low", replace(variant, threshold=0.04)),
        ("trigger_high", replace(variant, threshold=0.06)),
    ]
    rows = []
    for label, neighbour in settings:
        r = evaluate_variant(neighbour, tag=label)
        b = evaluate_variant(neighbour, overlay=False, tag=label + "_base")
        a, b = (
            metrics(score_sample(r, "validation"), cash_returns),
            metrics(score_sample(b, "validation"), cash_returns),
        )
        stable = (
            a["volatility"] < b["volatility"]
            and a["cvar_95"] > b["cvar_95"]
            and (a["cagr"] >= b["cagr"] - 0.015)
        )
        rows.append(
            {"trial_id": variant.trial_id, "neighbour": label, "stable": stable, **a}
        )
    path = RUN_DIR / "tables/parameter_plateau.csv"
    pd.DataFrame(rows).to_csv(path, mode="a", header=not path.exists(), index=False)
    return all(row["stable"] for row in rows)

In [ ]:
if MODE == "research":
    performance_rows, validation_rows, annual_rows, forecast_rows = (
        [],
        list(primary_failures),
        [],
        [],
    )
    for variant in variants():
        if variant.trial_id not in STRATEGY_RESULTS:
            continue
        strategy, matched_base = (
            STRATEGY_RESULTS[variant.trial_id],
            BASE_RESULTS[variant.trial_id],
        )
        for sample in ["development", "validation", "in_sample"]:
            for kind, frame in [("strategy", strategy), ("base", matched_base)]:
                performance_rows.append(
                    {
                        "trial_id": variant.trial_id,
                        "sample": sample,
                        "kind": kind,
                        **metrics(score_sample(frame, sample), cash_returns),
                    }
                )
        primary_pass = passes(
            metrics(score_sample(strategy, "validation"), cash_returns),
            metrics(score_sample(matched_base, "validation"), cash_returns),
        )
        print("Validation diagnostics:", variant.trial_id, flush=True)
        stress = evaluate_variant(variant, cost=10, tag="double_cost")
        stress_base = evaluate_variant(
            variant, cost=10, overlay=False, tag="double_cost_base"
        )
        cost_pass = passes(
            metrics(score_sample(stress, "validation"), cash_returns),
            metrics(score_sample(stress_base, "validation"), cash_returns),
        )
        forecasts = forecast_diagnostics(variant, strategy)
        forecast_rows.append(forecasts)
        rank_ic = forecasts.loc[forecasts["sample"] == "validation", "rank_ic"].iloc[0]
        yearly_passes = []
        for year, group in strategy.iloc[551:].groupby(strategy.index[551:].year):
            bm = matched_base.loc[group.index]
            sm, bm_metrics = (metrics(group, cash_returns), metrics(bm, cash_returns))
            annual_rows.append(
                {
                    "trial_id": variant.trial_id,
                    "year": year,
                    "sessions": len(group),
                    **{"strategy_" + k: value for k, value in sm.items()},
                    **{"base_" + k: value for k, value in bm_metrics.items()},
                }
            )
            if year >= 2019 and len(group) >= 240:
                yearly_passes.append(
                    sm["volatility"] < bm_metrics["volatility"]
                    and sm["cvar_95"] > bm_metrics["cvar_95"]
                )
        regime_pass = (
            len(yearly_passes) >= 2 and sum(yearly_passes) > len(yearly_passes) / 2
        )
        plateau_pass = neighbour_diagnostics(variant)
        gates = {
            "risk_CAGR_turnover": primary_pass,
            "double_cost": cost_pass,
            "regime": regime_pass,
            "forecast": rank_ic > 0,
            "neighbours": plateau_pass,
        }
        validation_rows.append(
            {
                "trial_id": variant.trial_id,
                "family": variant.family,
                "passed": all(gates.values()),
                **gates,
                "failure_reason": "; ".join(
                    (k for k, passed in gates.items() if not passed)
                ),
            }
        )
    PERFORMANCE = save_table("in_sample_performance", pd.DataFrame(performance_rows))
    validation = VALIDATION = save_table(
        "validation_results", pd.DataFrame(validation_rows)
    )
    save_table("annual_results", pd.DataFrame(annual_rows))
    save_table("forecast_diagnostics", pd.concat(forecast_rows, ignore_index=True))
    registry_snapshot = pd.read_csv(ROOT / "research/experiment_registry.csv")
    save_table("experiment_registry", registry_snapshot)
    print(
        "TOTAL STRATEGY VARIANTS TESTED =",
        registry_snapshot.loc[
            registry_snapshot["status"] == "started", "trial_id"
        ].nunique(),
    )
    display(validation)
    display(PERFORMANCE[PERFORMANCE["sample"] == "validation"])

## 15. Failure analysis
Failures remain visible, including data or optimiser failures. No winning candidate is guaranteed.

In [ ]:
if MODE == "research":
    failures = validation[~validation["passed"]]
    save_table("failed_and_rejected_variants", failures)
    display(failures)

## 16. Select and freeze strategy
Selection is Miles’s decision. Review diagnostics in the following cells before setting SELECTED_TRIAL and REVIEWED_RUN_ID. `study.freeze` reruns the fixed diagnostic set before binding all code/data/specification hashes. No automatic freeze from a favourable chart.

In [ ]:
if MODE == "research":
    passing = validation.loc[validation["passed"], "trial_id"].tolist()
    print("Eligible candidates:", passing)
    print("Set SELECTED_TRIAL and REVIEWED_RUN_ID only after evidence review.")

## 17. In-sample robustness
Inspect any pre-registered candidate without adopting it. Diagnostics include paired block bootstrap, 0/2/5/10 bp costs, static exposure estimated on development only, annual folds, stress windows and allocation attribution. Neighbouring parameters are evaluated by the runner; they cannot replace the registered candidate.

In [ ]:
def fast_stats(values, rates):
    wealth = np.r_[1.0, np.cumprod(1 + values)]
    downside = np.sqrt(np.mean(np.minimum(values - rates, 0) ** 2)) * np.sqrt(252)
    return {
        "volatility": np.std(values, ddof=1) * np.sqrt(252),
        "cagr": wealth[-1] ** (252 / len(values)) - 1,
        "sortino": np.mean(values - rates) * 252 / downside if downside else np.nan,
        "max_drawdown": np.min(wealth / np.maximum.accumulate(wealth) - 1),
        "cvar_95": np.mean(values[values <= np.quantile(values, 0.05)]),
    }


def bootstrap(a, b, rf, repetitions=1000):
    """Paired moving blocks preserve within-block dependence; paths rebuilt."""
    rng = np.random.default_rng(42)
    x, y, rates = (
        a.net_return.to_numpy(),
        b.net_return.to_numpy(),
        rf.reindex(a.index).to_numpy(),
    )
    rows = []
    n = len(x)
    for block in (5, 10, 20):
        samples = []
        for _ in range(repetitions):
            starts = rng.integers(0, n - block + 1, size=int(np.ceil(n / block)))
            ix = np.concatenate([np.arange(start, start + block) for start in starts])[
                :n
            ]
            first, second = fast_stats(x[ix], rates[ix]), fast_stats(y[ix], rates[ix])
            samples.append({key: first[key] - second[key] for key in first})
        frame = pd.DataFrame(samples)
        for key in frame:
            rows.append(
                {
                    "block_sessions": block,
                    "metric": key,
                    "lower_95": frame[key].quantile(0.025),
                    "upper_95": frame[key].quantile(0.975),
                    "repetitions": repetitions,
                    "seed": 42,
                }
            )
    return pd.DataFrame(rows)

In [ ]:
INSPECT_TRIAL = "E7"
if MODE == "research" and INSPECT_TRIAL in STRATEGY_RESULTS:
    selected_variant = next(v for v in variants() if v.trial_id == INSPECT_TRIAL)
    inspected = score_sample(STRATEGY_RESULTS[INSPECT_TRIAL], "in_sample")
    inspected_base = score_sample(BASE_RESULTS[INSPECT_TRIAL], "in_sample")
    paired = pd.read_csv(
        RUN_DIR / f"{INSPECT_TRIAL}-forecast-pairs.csv", index_col=0, parse_dates=True
    )
    display(
        pd.read_csv(RUN_DIR / "tables/forecast_diagnostics.csv").query(
            "trial_id == @INSPECT_TRIAL"
        )
    )
    figure(
        "inspected_forecast_vs_future",
        paired,
        "Annualised forecast / future 21-session volatility",
    )
    ax = paired.plot.scatter(x="forecast", y="future", alpha=0.15, figsize=(7, 5))
    ax.set_title("IS forecast calibration; future volatility is a diagnostic target")
    ax.figure.tight_layout()
    ax.figure.savefig(RUN_DIR / "figures/calibration.png", dpi=160)
    plt.show()
    equity = (
        1
        + pd.DataFrame(
            {"base": inspected_base.net_return, "targeted": inspected.net_return}
        )
    ).cumprod()
    figure("inspected_equity", equity, "Net wealth; 5 bp per risky purchase/sale")
    figure(
        "inspected_drawdown",
        equity / equity.cummax().clip(lower=1) - 1,
        "Drawdown fraction",
    )
    figure(
        "inspected_exposure",
        inspected[["raw_exposure", "exposure"]],
        "Risky NAV fraction",
    )
    figure(
        "inspected_base_weights",
        inspected.attrs["base_weights"].loc[inspected.index],
        "Base allocation fraction",
    )
    figure(
        "inspected_targeted_weights",
        inspected.attrs["weights"].loc[inspected.index],
        "Risky NAV fraction",
    )
    confidence_intervals = save_table(
        "bootstrap_confidence_intervals",
        bootstrap(inspected, inspected_base, cash_returns),
    )
    display(confidence_intervals)
    cost_rows = []
    for cost in [0, 2, 5, 10]:
        frame = evaluate_variant(selected_variant, cost=cost, tag=f"cost_{cost}")
        cost_rows.append(
            {
                "cost_bps": cost,
                **metrics(score_sample(frame, "in_sample"), cash_returns),
            }
        )
    cost_sensitivity = save_table("cost_sensitivity", pd.DataFrame(cost_rows))
    display(cost_sensitivity)
    figure(
        "cost_sensitivity",
        cost_sensitivity.set_index("cost_bps")[["cagr", "volatility"]],
        "Annualised fraction",
    )
    development_exposure = score_sample(
        STRATEGY_RESULTS[INSPECT_TRIAL], "development"
    ).exposure.mean()
    static = evaluate_variant(
        selected_variant, static=float(development_exposure), tag="static_equivalent"
    )
    static_comparison = save_table(
        "static_equivalent_comparison",
        pd.DataFrame(
            [
                {"kind": name, **metrics(frame, cash_returns)}
                for name, frame in [
                    ("targeted", inspected),
                    ("base", inspected_base),
                    ("static", score_sample(static, "in_sample")),
                ]
            ]
        ),
    )
    display(static_comparison)

## 18. Risk management
Reports include gross exposure, asset weights, financing, tails, unrecovered drawdowns, cash/overlay attribution and descriptive factor/downside betas. Rolling Sharpe and Sortino are descriptive only.

In [ ]:
if MODE == "research" and INSPECT_TRIAL in STRATEGY_RESULTS:
    display(
        pd.Series(metrics(inspected, cash_returns), name="net risk metrics").to_frame()
    )
    figure(
        "annual_turnover",
        inspected.turnover.groupby(inspected.index.year).sum().to_frame(),
        "Half-L1 turnover including cash",
    )
    figure(
        "cost_drag",
        inspected[["transaction_cost", "financing_cost"]].cumsum(),
        "Cumulative NAV fraction",
    )
    figure(
        "annual_comparison",
        (
            1
            + pd.DataFrame(
                {"base": inspected_base.net_return, "targeted": inspected.net_return}
            )
        )
        .groupby(inspected.index.year)
        .prod()
        - 1,
        "Calendar-year net return",
    )

## 19. Liquidity and capacity
Prior-session 20-day median raw dollar volume; drift-adjusted trade dollars. AUM $1m–$1bn, square-root impact Y=.25/.5/1. These are rough stress estimates, not calibrated execution costs. Capacity thresholds are undefined if there is no positive CAGR benefit.

In [ ]:
def capacity(variant, result):
    assets = list(variant.universe)
    trades = result.attrs["trades"].loc[result.index, assets].abs()
    active = trades.to_numpy() > 1e-12
    adv = (
        (RAW_CLOSE[assets] * volumes[assets])
        .rolling(20)
        .median()
        .shift(1)
        .loc[result.index]
    )
    sigma = (
        prices[assets]
        .pct_change(fill_method=None)
        .rolling(20)
        .std()
        .shift(1)
        .loc[result.index]
    )
    if ((adv <= 0) & (trades > 0)).any().any() or (
        adv.isna() & (trades > 0)
    ).any().any():
        raise ValueError("No prior-session liquidity for executed trades")
    rows = []
    for aum in (1000000.0, 10000000.0, 100000000.0, 500000000.0, 1000000000.0):
        initial_nav = result.equity.iloc[0] / (1 + result.net_return.iloc[0])
        prior_nav = result.equity.shift(1).fillna(initial_nav) / initial_nav
        before_fee_nav = prior_nav * (1 + result.gross_return)
        participation = trades.mul(before_fee_nav * aum, axis=0) / adv
        p = participation.to_numpy()[active]
        for impact_y in (0.25, 0.5, 1.0):
            impact = impact_y * sigma * np.sqrt(participation)
            drag = (trades * impact).sum(axis=1)
            adjusted = result.copy()
            adjusted.attrs = {}
            adjusted["net_return"] -= drag
            m = metrics(adjusted, cash_returns)
            worst = participation.max().idxmax()
            rows.append(
                {
                    "aum_usd": aum,
                    "impact_y": impact_y,
                    "median_participation": np.median(p) if len(p) else 0,
                    "p95_participation": np.quantile(p, 0.95) if len(p) else 0,
                    "maximum_participation": np.max(p) if len(p) else 0,
                    "worst_asset": worst,
                    "trades_above_1pct": np.mean(p > 0.01) if len(p) else 0,
                    "trades_above_5pct": np.mean(p > 0.05) if len(p) else 0,
                    "annual_impact_drag": drag.sum() * 252 / len(result),
                    "annual_total_cost_drag": m["implementation_cost"]
                    + drag.sum() * 252 / len(result),
                    "net_sharpe": m["sharpe"],
                    "net_cagr": m["cagr"],
                    "weighted_impact_bps": drag.sum() / trades.sum().sum() * 10000
                    if trades.sum().sum()
                    else 0,
                }
            )
    return pd.DataFrame(rows)


def export_evidence(trial_id, final=False):
    variant = next(v for v in variants() if v.trial_id == trial_id)
    full = STRATEGY_RESULTS[trial_id]
    base_full = BASE_RESULTS[trial_id]
    report_sample = "OOS" if final else "in_sample"
    result, base = (
        score_sample(full, report_sample),
        score_sample(base_full, report_sample),
    )
    for key in ("weights", "base_weights", "trades", "orders"):
        result.attrs[key] = full.attrs[key]
    avg = (
        FROZEN["static_exposure"]
        if final
        else float(score_sample(full, "development").exposure.mean())
    )
    static = evaluate_variant(variant, static=avg, tag="static_equivalent")
    static = score_sample(static, report_sample)
    comparison = pd.DataFrame(
        [
            {"kind": kind, **metrics(frame, cash_returns)}
            for kind, frame in [
                ("strategy", result),
                ("base", base),
                ("static", static),
            ]
        ]
    ).set_index("kind")
    save_table("static_equivalent_comparison", comparison.reset_index())
    saved = pd.read_csv(RUN_DIR / "tables/static_equivalent_comparison.csv").set_index(
        "kind"
    )
    for column in ("cagr", "volatility", "max_drawdown", "turnover"):
        assert np.allclose(saved[column], comparison[column], equal_nan=True)
    incremental = comparison.loc["strategy"].drop("unrecovered_at_end").astype(
        float
    ) - comparison.loc["base"].drop("unrecovered_at_end").astype(float)
    save_table(
        "incremental_metrics", incremental.rename("strategy_minus_base").reset_index()
    )
    save_table("bootstrap_confidence_intervals", bootstrap(result, base, cash_returns))
    asset_returns = prices.pct_change(fill_method=None).loc[result.index]
    factors = ["SPY", "TLT", "GLD"] + (
        ["QQQ", "HYG", "DBC"] if len(variant.universe) == 6 else []
    )
    available = asset_returns[factors].notna().all(axis=1)
    x = np.column_stack(
        [np.ones(available.sum()), asset_returns.loc[available, factors].to_numpy()]
    )
    betas = np.linalg.lstsq(x, result.net_return.loc[available].to_numpy(), rcond=None)[
        0
    ]
    down = base.net_return < 0
    downside_beta = (
        result.net_return[down].cov(base.net_return[down]) / base.net_return[down].var()
        if base.net_return[down].var()
        else np.nan
    )
    save_table(
        "factor_exposures",
        pd.DataFrame(
            {
                "factor": ["intercept_daily", *factors, "downside_matched_base"],
                "coefficient": [*betas, downside_beta],
            }
        ),
    )
    weights = result.attrs["weights"].loc[result.index]
    base_weights = result.attrs["base_weights"].loc[result.index]
    equity = pd.DataFrame(
        {
            kind: (1 + frame.net_return).cumprod()
            for kind, frame in [
                ("base", base),
                ("targeted", result),
                ("static", static),
            ]
        }
    )
    drawdown = equity / equity.cummax().clip(lower=1) - 1
    figures = {
        "asset_total_return_histories": (
            (1 + asset_returns).cumprod(),
            "Wealth per initial dollar",
        ),
        "raw_target_exposure": (result[["raw_exposure"]], "Risky NAV fraction"),
        "actual_exposure": (result[["exposure"]], "Risky NAV fraction"),
        "base_weights": (base_weights, "Base allocation fraction"),
        "targeted_weights": (weights, "Risky NAV fraction"),
        "equity_curves": (equity, "Wealth per initial dollar; net 5 bp"),
        "drawdown_curves": (drawdown, "Drawdown fraction"),
        "rolling_realised_vol": (
            pd.DataFrame(
                {
                    "20d": result.net_return.rolling(20).std() * np.sqrt(252),
                    "63d": result.net_return.rolling(63).std() * np.sqrt(252),
                }
            ),
            "Annualised volatility",
        ),
        "target_vs_realised_vol": (
            pd.DataFrame(
                {
                    "target": variant.target,
                    "realised_63d": result.net_return.rolling(63).std() * np.sqrt(252),
                },
                index=result.index,
            ),
            "Annualised volatility",
        ),
        "rolling_sharpe_sortino": (
            pd.DataFrame(
                {
                    "sharpe": (result.net_return - cash_returns).rolling(252).mean()
                    / result.net_return.rolling(252).std()
                    * np.sqrt(252),
                    "sortino": (result.net_return - cash_returns).rolling(252).mean()
                    / np.minimum(result.net_return - cash_returns, 0)
                    .pow(2)
                    .rolling(252)
                    .mean()
                    .pow(0.5)
                    * np.sqrt(252),
                }
            ),
            "Descriptive trailing ratio",
        ),
        "annual_turnover": (
            result.turnover.groupby(result.index.year).sum().to_frame(),
            "Half-L1 turnover including cash",
        ),
        "transaction_cost_drag": (
            result[["transaction_cost", "financing_cost"]].cumsum(),
            "Cumulative NAV fraction",
        ),
        "calendar_year_returns": (
            (1 + pd.DataFrame({"base": base.net_return, "strategy": result.net_return}))
            .groupby(result.index.year)
            .prod()
            - 1,
            "Net return fraction",
        ),
        "contribution_decomposition": (
            result[
                [
                    "risky_contribution",
                    "cash_contribution",
                    "financing_cost",
                    "transaction_cost",
                ]
            ].cumsum(),
            "Cumulative arithmetic contribution",
        ),
    }
    for name, (data, ylabel) in figures.items():
        figure(name, data, ylabel)
    contribution = (
        result[
            [
                "risky_contribution",
                "cash_contribution",
                "financing_cost",
                "transaction_cost",
            ]
        ]
        .sum()
        .to_dict()
    )
    contribution["base_allocation_return_sum"] = base.net_return.sum()
    contribution["overlay_incremental_return_sum"] = (
        result.net_return - base.net_return
    ).sum()
    save_table("contribution_decomposition", pd.DataFrame([contribution]))
    stress_rows = []
    episodes = [
        ("COVID", "2020-02-19", "2020-04-30"),
        ("inflation_rates", "2022-01-01", "2022-12-31"),
    ]
    worst = (1 + base.net_return).rolling(20).apply(np.prod, raw=True) - 1
    used = set()
    for end in worst.sort_values().index:
        pos = base.index.get_loc(end)
        positions = set(range(pos - 19, pos + 1))
        if positions & used or pos < 19:
            continue
        episodes.append(
            (
                f"base_worst_{len(episodes) - 1}",
                str(base.index[pos - 19].date()),
                str(end.date()),
            )
        )
        used |= positions
        if len(episodes) >= 7:
            break
    for label, start, end in episodes:
        for kind, frame in [("strategy", result), ("base", base)]:
            portion = frame.loc[start:end]
            if len(portion) > 1:
                stress_rows.append(
                    {
                        "episode": label,
                        "start": start,
                        "end": end,
                        "kind": kind,
                        **metrics(portion, cash_returns),
                    }
                )
    stress = save_table("stress_period_results", pd.DataFrame(stress_rows))
    if not stress.empty:
        figure(
            "worst_stress_windows",
            stress.pivot(index="episode", columns="kind", values="max_drawdown"),
            "Drawdown fraction",
        )
    save_table(
        "turnover_breakdown",
        pd.DataFrame(
            {
                "year": result.index.year,
                "turnover": result.turnover,
                "traded_notional": result.traded_notional,
                "cost": result.transaction_cost,
            }
        )
        .groupby("year")
        .sum()
        .reset_index(),
    )
    capacities = save_table("liquidity_capacity", capacity(variant, result))
    for name, col in [
        ("capacity_participation", "p95_participation"),
        ("net_sharpe_vs_aum", "net_sharpe"),
        ("cost_drag_vs_aum", "annual_total_cost_drag"),
    ]:
        figure(
            name,
            capacities.pivot(index="aum_usd", columns="impact_y", values=col),
            col.replace("_", " "),
        )
    benefit = comparison.loc["strategy", "cagr"] - comparison.loc["base", "cagr"]
    primary = capacities[capacities.impact_y == 0.5]
    q = primary.p95_participation.iloc[0]
    estimates = {
        "aum_at_1pct_p95_adv": primary.aum_usd.iloc[0] * 0.01 / q if q > 0 else np.nan,
        "aum_half_benefit": next(
            iter(primary.loc[primary.annual_impact_drag >= benefit / 2, "aum_usd"]),
            np.nan,
        )
        if benefit > 0
        else np.nan,
        "aum_no_benefit": next(
            iter(
                primary.loc[
                    primary.net_cagr <= comparison.loc["base", "cagr"], "aum_usd"
                ]
            ),
            np.nan,
        )
        if benefit > 0
        else np.nan,
        "interpretation": "Rough stress-model thresholds; undefined if no positive CAGR benefit; grid upper bounds",
    }
    save_table("capacity_estimates", pd.DataFrame([estimates]))
    limits = pd.DataFrame(
        [
            {
                "rule": "gross_risky_exposure",
                "limit": variant.cap,
                "meaning": "at trade; drift reported",
            },
            {
                "rule": "base_asset_cap",
                "limit": 0.35 if len(variant.universe) == 6 else 0.7,
                "meaning": "base allocation, leverage scales cap",
            },
            {"rule": "de_risk", "limit": variant.threshold, "meaning": variant.control},
            {
                "rule": "cost_bps",
                "limit": 5,
                "meaning": "per risky dollar bought or sold",
            },
            {
                "rule": "borrow_spread_bps",
                "limit": 50,
                "meaning": "ACT/365 over initial-release cash rate",
            },
        ]
    )
    save_table("risk_management_summary", limits)
    if not final:
        sensitivity = []
        for cost in (0, 2, 5, 10):
            r = evaluate_variant(variant, cost=cost, tag=f"cost_{cost}")
            sensitivity.append(
                {
                    "cost_bps": cost,
                    **metrics(score_sample(r, "in_sample"), cash_returns),
                }
            )
        costs = save_table("cost_sensitivity", pd.DataFrame(sensitivity))
        figure(
            "cost_sensitivity",
            costs.set_index("cost_bps")[["cagr", "volatility"]],
            "Annualised fraction",
        )
        if variant.cap > 1:
            save_table(
                "financing_sensitivity",
                pd.DataFrame(
                    [
                        {
                            "borrow_bps": spread,
                            **metrics(
                                score_sample(
                                    evaluate_variant(
                                        variant, borrow=spread, tag=f"borrow_{spread}"
                                    ),
                                    "in_sample",
                                ),
                                cash_returns,
                            ),
                        }
                        for spread in (0, 50, 100)
                    ]
                ),
            )
        plateau = pd.read_csv(RUN_DIR / "tables/parameter_plateau.csv")
        figure(
            "validation_plateau",
            plateau[plateau.trial_id == trial_id].set_index("neighbour")[
                ["cagr", "volatility"]
            ],
            "Validation annualised fraction",
        )
        paired = pd.read_csv(
            RUN_DIR / f"{trial_id}-forecast-pairs.csv", index_col=0, parse_dates=True
        )
        figure(
            "forecast_vs_future_vol", paired, "Annualised volatility; next 21 sessions"
        )
        ax = paired.plot.scatter(x="forecast", y="future", figsize=(7, 5), alpha=0.15)
        ax.set_title("Forecast calibration: in-sample only")
        ax.figure.tight_layout()
        ax.figure.savefig(RUN_DIR / "figures/forecast_calibration.png", dpi=160)
        plt.show()
        plt.close(ax.figure)
    summary = (
        f"# Volatility targeting: {trial_id}\n\nSample: {report_sample}. Hypothesis: persistent variance permits causal risk sizing.\nData: {DATA_METADATA['source']}; hashes in data_provenance.json.\nCosts: 5 bp per risky purchase/sale; borrowing cash rate +50 bp.\nMethod: lagged close signals, next-close fills, drifted holdings.\n\n"
        + comparison.to_csv()
        + "\n\nFailures: see failed_and_rejected_variants.csv in the research run.\nIS and OOS evidence must be cited separately; unavailable OOS is not zero.\nRisk, capacity and limitations: risk_management_summary.csv, liquidity_capacity.csv, HYPOTHESIS.md.\nThese are modelled closing fills and rough impact stresses; no live execution claim.\n"
    )
    (RUN_DIR / "note-ready-summary.md").write_text(summary)
    return comparison


def freeze_candidate(trial_id, expected_research_run):
    if MODE != "research" or OOS_FLAG.exists():
        raise RuntimeError("Cannot freeze after OOS access")
    if expected_research_run != RUN_ID:
        raise ValueError("Review and supply the exact research run ID")
    validation = VALIDATION.set_index("trial_id")
    if trial_id not in validation.index or not validation.loc[trial_id, "passed"]:
        raise ValueError("Only candidates passing every validation gate may freeze")
    export_evidence(trial_id)
    variant = next(v for v in variants() if v.trial_id == trial_id)
    exposure = float(
        score_sample(STRATEGY_RESULTS[trial_id], "development").exposure.mean()
    )
    frozen = {
        "variant": variant.parameters(),
        "hypothesis_hash": HYPOTHESIS_HASH,
        "code_hash": code_hash(ROOT),
        "registry_hash": sha(ROOT / "research/experiment_registry.csv"),
        "research_data": {
            **DATA_METADATA,
            "paths": [
                str(Path(path).relative_to(ROOT)) for path in DATA_METADATA["paths"]
            ],
        },
        "research_run": RUN_ID,
        "oos_start": str(boundary["oos_start"].date()),
        "static_exposure": exposure,
        "cost_bps": 5.0,
        "borrow_bps": 50.0,
        "seed": 42,
    }
    path = ROOT / "research/frozen_strategy.json"
    with path.open("x") as handle:
        json.dump(frozen, handle, indent=2, sort_keys=True)
    digest = sha(path)
    (ROOT / "research/frozen_strategy.sha256").write_text(digest + "\n")
    save_table("frozen_strategy", pd.json_normalize(frozen))
    print("Frozen SHA-256:", digest)
    return digest

In [ ]:
if MODE == "research" and INSPECT_TRIAL in STRATEGY_RESULTS:
    capacity_table = save_table(
        "liquidity_capacity", capacity(selected_variant, inspected)
    )
    display(capacity_table)
    figure(
        "capacity_participation",
        capacity_table.pivot(
            index="aum_usd", columns="impact_y", values="p95_participation"
        ),
        "95th percentile fraction of lagged ADV",
    )
    figure(
        "capacity_net_sharpe",
        capacity_table.pivot(index="aum_usd", columns="impact_y", values="net_sharpe"),
        "Impact-adjusted net Sharpe",
    )
    display(export_evidence(INSPECT_TRIAL))
if MODE == "research" and SELECTED_TRIAL is not None:
    freeze_candidate(SELECTED_TRIAL, REVIEWED_RUN_ID)

## 20. FINAL OOS GATE
This reserves access before any holdout request. A failed acquisition consumes the reservation and requires explicit review; never delete the flag to pretend the holdout is untouched. External expected hash is required in addition to the saved hash. Final mode bypasses every search cell.

In [ ]:
def unlock_final(expected_hash):
    global FROZEN
    if MODE != "final" or not expected_hash:
        raise RuntimeError(
            "Final mode requires an externally supplied expected SHA-256"
        )
    path = ROOT / "research/frozen_strategy.json"
    if (
        sha(path) != expected_hash
        or (ROOT / "research/frozen_strategy.sha256").read_text().strip()
        != expected_hash
    ):
        raise ValueError("Frozen strategy hash mismatch")
    frozen = json.loads(path.read_text())
    if (
        frozen["code_hash"] != code_hash(ROOT)
        or frozen["hypothesis_hash"] != HYPOTHESIS_HASH
    ):
        raise ValueError("Code or hypothesis changed after freeze")
    if frozen["registry_hash"] != sha(ROOT / "research/experiment_registry.csv"):
        raise ValueError("Trial registry changed after freeze")
    if frozen["oos_start"] != str(boundary["oos_start"].date()):
        raise ValueError("Holdout boundary changed")
    if OOS_FLAG.exists():
        raise RuntimeError(WARNING)
    for path_string, key in zip(
        frozen["research_data"]["paths"], ["market_sha256", "rates_sha256"]
    ):
        if sha(ROOT / path_string) != frozen["research_data"][key]:
            raise ValueError("Frozen research cache missing or changed")
    print(
        "FINAL OOS ACCESS: strategy frozen; no further research or retuning permitted."
    )
    with OOS_FLAG.open("x") as handle:
        json.dump(
            {
                "opened_utc": datetime.now(UTC).isoformat(),
                "frozen_hash": expected_hash,
                "run_id": RUN_ID,
            },
            handle,
        )
        handle.flush()
        os.fsync(handle.fileno())
    FROZEN = frozen
    return frozen

In [ ]:
if MODE == "final":
    unlock_final(os.environ.get("GQH_FROZEN_SHA256"))
else:
    print("OOS LOCKED — research mode")

## 21. Acquire OOS data
Only after exclusive access reservation. Frozen IS cache is retained; no revised IS redownload. The full provider cache remains ignored.

In [ ]:
if MODE == "final":
    load_market_data(final=True)

## 22. Evaluate frozen strategy once
Exactly the frozen strategy, matched base and frozen static comparator. No search or parameter sensitivity in final mode.

In [ ]:
def evaluate_final():
    if MODE != "final" or not globals().get("FROZEN"):
        raise RuntimeError("Final gate required")
    parameters = FROZEN["variant"].copy()
    parameters["universe"] = tuple(parameters["universe"])
    variant = Variant(**parameters)
    result = evaluate_variant(variant, tag="official_oos")
    base = evaluate_variant(variant, overlay=False, tag="official_oos_base")
    STRATEGY_RESULTS[variant.trial_id], BASE_RESULTS[variant.trial_id] = (result, base)
    save_table(
        "oos_performance",
        pd.DataFrame(
            [
                {"kind": kind, **metrics(score_sample(r, "OOS"), cash_returns)}
                for kind, r in [("strategy", result), ("base", base)]
            ]
        ),
    )
    export_evidence(variant.trial_id, final=True)
    (RUN_DIR / "oos_result_preserved.flag").write_text(
        "Official result preserved; no retuning.\n"
    )
    return score_sample(result, "OOS")

In [ ]:
if MODE == "final":
    official = evaluate_final()

## 23. OOS report
Official OOS metrics are saved separately before any illustrative full-history curve.

In [ ]:
if MODE == "final":
    display(pd.read_csv(RUN_DIR / "tables/oos_performance.csv"))

## 24. Full-history illustrative curve
FULL-HISTORY VISUALISATION — NOT USED FOR MODEL SELECTION. Available only after the official OOS result is preserved.

In [ ]:
if MODE == "final":
    assert (RUN_DIR / "oos_result_preserved.flag").exists()

    result = next(iter(STRATEGY_RESULTS.values()))
    figure(
        "full_history_illustration",
        (1 + result[["net_return"]]).cumprod(),
        "Net wealth per dollar",
        "FULL-HISTORY VISUALISATION — NOT USED FOR MODEL SELECTION",
    )

## 25. Note-ready summary and submission checklist
Saved numbers must be used directly in the paper. No claim of completion or validated edge is made by authoring this notebook.

In [ ]:
summary_path = RUN_DIR / "note-ready-summary.md"
if summary_path.exists():
    print(summary_path.read_text())
print(
    "\n".join(
        "[ ] " + item
        for item in [
            "Quant note <=5 pages, 11 pt and standard margins",
            "Hypothesis committed before results; all variants and failures disclosed",
            "Separate IS/OOS net Sharpe, drawdown, turnover and equity",
            "Risk, costs, capacity and source limitations reviewed",
            "Public repository, pinned dependencies and reproduction command",
            "No credentials or licensed raw data committed",
            "One official OOS evaluation after explicit freeze",
            "All team members listed on Devpost",
        ]
    )
)